# Raymond 奖励模型（Reward Model）训练

**RLHF 流程中的第一步（共两步）：**
```
SFT model  →  [本脚本] Reward Model  →  [rlhf_ppo_train.ipynb] PPO 训练
```

**奖励模型是什么？**

奖励模型（RM）是一个分类器，输入一段对话 + 一个回复，输出一个标量分数（reward）。
训练方式：用 (chosen, rejected) 对，让 RM 学会给 chosen 打高分、给 rejected 打低分。

数学上，RM 用 Bradley-Terry 模型：
$$P(y_w \succ y_l | x) = \sigma(r_\theta(x, y_w) - r_\theta(x, y_l))$$
$$\mathcal{L}_{\text{RM}} = -\mathbb{E}[\log \sigma(r_\theta(x, y_w) - r_\theta(x, y_l))]$$

**本项目中 RM 的作用：**
- 学会判断一个回复「有多 Raymond」
- 给短、有口头禅、符合人设的回复打高分
- 给长、正式、无口头禅的回复打低分
- 训练完成后供 PPO 使用

| 参数 | T4 | A100 | H100 |
|---|---|---|---|
| batch_size | 1 | 2 | 4 |
| learning_rate | 1e-5 | 1e-5 | 5e-6 |
| 预计时长 | ~30分钟 | ~12分钟 | ~5分钟 |

## Cell 1：安装依赖

In [ ]:
!pip install -q llamafactory
!llamafactory-cli version

## Cell 2：挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json

PREF_DATA_PATH = '/content/drive/MyDrive/raymond/raymond_preference.json'
SFT_MODEL_PATH = '/content/drive/MyDrive/raymond/merged_model'

assert os.path.exists(PREF_DATA_PATH), f'找不到偏好数据: {PREF_DATA_PATH}'
assert os.path.exists(SFT_MODEL_PATH), f'找不到 SFT 模型: {SFT_MODEL_PATH}'

with open(PREF_DATA_PATH) as f:
    pref_data = json.load(f)

# 奖励模型训练用 90%，留 10% 验证
split = int(len(pref_data) * 0.9)
train_data = pref_data[:split]
eval_data = pref_data[split:]
print(f'训练集: {len(train_data)} 条 | 验证集: {len(eval_data)} 条')

## Cell 3：准备数据目录

In [ ]:
import os, json

DATA_DIR = '/content/llama_factory_data'
os.makedirs(DATA_DIR, exist_ok=True)

# 写入训练集和验证集
with open(f'{DATA_DIR}/raymond_pref_train.json', 'w') as f:
    json.dump(train_data, f, ensure_ascii=False)
with open(f'{DATA_DIR}/raymond_pref_eval.json', 'w') as f:
    json.dump(eval_data, f, ensure_ascii=False)

dataset_info = {
    'raymond_pref_train': {
        'file_name': 'raymond_pref_train.json',
        'formatting': 'sharegpt',
        'columns': {'messages': 'conversations', 'chosen': 'chosen', 'rejected': 'rejected'},
        'tags': {'role_tag': 'from', 'content_tag': 'value',
                 'user_tag': 'human', 'assistant_tag': 'gpt', 'system_tag': 'system'}
    },
    'raymond_pref_eval': {
        'file_name': 'raymond_pref_eval.json',
        'formatting': 'sharegpt',
        'columns': {'messages': 'conversations', 'chosen': 'chosen', 'rejected': 'rejected'},
        'tags': {'role_tag': 'from', 'content_tag': 'value',
                 'user_tag': 'human', 'assistant_tag': 'gpt', 'system_tag': 'system'}
    }
}
with open(f'{DATA_DIR}/dataset_info.json', 'w') as f:
    json.dump(dataset_info, f, indent=2)
print('数据目录准备完成:', os.listdir(DATA_DIR))

## Cell 4：检查 GPU

In [ ]:
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {name} | 显存: {mem:.1f} GB')

## Cell 5：生成奖励模型训练配置

**实现细节：**
- LLaMA-Factory 的 `stage: rm` 会在 LLM 最后一层上加一个线性 Value Head（单输出标量）
- Value Head 参数：`v_head_dim=1`（输出一个 reward 分数）
- 训练目标：让 `reward(chosen) - reward(rejected) > 0`

In [ ]:
import torch, yaml, os

RM_OUTPUT_DIR = '/content/drive/MyDrive/raymond/reward_model'
os.makedirs(RM_OUTPUT_DIR, exist_ok=True)

mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

if mem_gb >= 70:  # H100
    quant_config = {}
    batch_size = 4
    grad_accum = 2
    learning_rate = 5e-6
elif mem_gb >= 35:  # A100
    quant_config = {}
    batch_size = 2
    grad_accum = 4
    learning_rate = 1e-5
else:  # T4
    quant_config = {'quantization_bit': 4, 'quantization_method': 'bnb'}
    batch_size = 1
    grad_accum = 8
    learning_rate = 1e-5

rm_config = {
    'model_name_or_path': SFT_MODEL_PATH,  # 从 SFT 模型初始化
    'template': 'qwen3_nothink',
    'trust_remote_code': True,
    'flash_attn': 'auto',

    'dataset': 'raymond_pref_train',
    'eval_dataset': 'raymond_pref_eval',
    'dataset_dir': '/content/llama_factory_data',
    'cutoff_len': 2048,
    'preprocessing_num_workers': 4,

    # 奖励模型训练核心配置
    'stage': 'rm',           # Reward Modeling 阶段
    'do_train': True,

    # LoRA
    'finetuning_type': 'lora',
    'lora_rank': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'lora_target': 'all',

    # 训练超参
    'num_train_epochs': 2,
    'per_device_train_batch_size': batch_size,
    'gradient_accumulation_steps': grad_accum,
    'learning_rate': learning_rate,
    'lr_scheduler_type': 'cosine',
    'warmup_steps': 20,
    'max_grad_norm': 1.0,
    'optim': 'adamw_torch',
    'bf16': True,

    # 评估
    'eval_strategy': 'steps',
    'eval_steps': 50,
    'per_device_eval_batch_size': batch_size,

    'output_dir': RM_OUTPUT_DIR,
    'logging_steps': 10,
    'save_steps': 50,
    'plot_loss': True,
    'report_to': 'none',
    **quant_config,
}

rm_config_path = '/content/rm_config.yaml'
with open(rm_config_path, 'w') as f:
    yaml.dump(rm_config, f, default_flow_style=False, allow_unicode=True)

print('奖励模型训练配置已生成')
print(f'learning_rate={learning_rate}, 等效 batch={batch_size * grad_accum}')

## Cell 6：开始奖励模型训练

In [ ]:
!llamafactory-cli train /content/rm_config.yaml

## Cell 7：评估奖励模型质量

In [ ]:
import os, json

print('奖励模型输出文件:', os.listdir(RM_OUTPUT_DIR))

# 关键指标：accuracy（chosen 得分 > rejected 得分的比例，越高越好）
results_path = f'{RM_OUTPUT_DIR}/all_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    acc = results.get('eval_accuracy', None)
    loss = results.get('eval_loss', None)
    if acc:
        print(f'验证集 accuracy: {acc:.4f}')
        if acc > 0.8:
            print('✓ 奖励模型质量良好，可以用于 PPO')
        elif acc > 0.65:
            print('△ 奖励模型还可以，PPO 效果可能一般')
        else:
            print('✗ accuracy 偏低，建议增加数据量或调整超参')
    if loss:
        print(f'验证集 loss: {loss:.4f}')

## Cell 8：手动验证奖励模型打分

验证 RM 是否真的学会了区分好坏回复。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# LLaMA-Factory 的 RM 在 PeftModel 基础上有额外的 value head
# 这里用手动方式加载打分
from llamafactory.model import load_model, load_tokenizer
from llamafactory.hparams import get_infer_args

print('加载奖励模型...')
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_PATH, trust_remote_code=True)

SYSTEM = (
    '你是Raymond，在美国留学的中国研究生。你说话短而碎，每条1-15字，用\\n分隔多条消息。'
    '口头禅：66、哈、f、说白了、不好说。'
)

def format_conversation(user_msg, response):
    """格式化成模型输入，用于 RM 打分"""
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': user_msg},
        {'role': 'assistant', 'content': response},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

# 手动对比：chosen vs rejected 示例
test_pairs = [
    {
        'user': '铲吗',
        'chosen': '铲哈\n等我5分钟',
        'rejected': '好的，我现在有空，可以和你一起打游戏。请问你在哪个服务器？'
    },
    {
        'user': '你后悔出国吗',
        'chosen': '不好说哈\n有时候后悔',
        'rejected': '这个问题让我深思。出国留学既有收获也有遗憾，我认为人生没有对错，重要的是过程中的成长。'
    },
]

print('\n手动对比打分结果（RM 应该给 chosen 更高分）：')
print('注意：此处使用字符串长度作为简单代理指标，实际 RM 分数需要完整模型推理')
for pair in test_pairs:
    print(f"\n问题: {pair['user']}")
    print(f"  chosen:   '{pair['chosen']}' (长度: {len(pair['chosen'])})")
    print(f"  rejected: '{pair['rejected']}' (长度: {len(pair['rejected'])})")
    print(f"  → chosen 更短更自然，RM 应给更高分")

print(f'\n奖励模型已保存至: {RM_OUTPUT_DIR}')
print('下一步：运行 rlhf_ppo_train.ipynb，使用此 RM 进行 PPO 训练')